In [ ]:
from __future__ import annotations
from IPython.display import Image, display
from langgraph.graph import StateGraph, START, END
from langgraph.graph import MessagesState
from langgraph.prebuilt import ToolNode
from langgraph.prebuilt import tools_condition

import random 
from typing import Literal
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END

from typing import TypedDict, List
import argparse
import json
import os
import sys
from pathlib import Path
from typing import Optional
from pathlib import Path
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple
# Allows running this script from the repository root without installing
# the package in editable mode.
#PROJECT_ROOT = Path(__file__).resolve().parent
PROJECT_ROOT = Path("C:/self-study/ai/build/repository-intelligence/")
SRC_ROOT = PROJECT_ROOT / "src"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

from repo_intelligence.llm.config import load_env  # noqa: E402
from repo_intelligence.langgraph.planner import get_plan
from repo_intelligence.orchestration.roles import RoleAssignment

from repo_intelligence.planning import (  # noqa: E402
    InvestigationPlan,
    PlanValidationError,
    build_planning_request,
    plan_from_response,
    plan_to_dict,
    validate_plan,
)
from repo_intelligence.agent import (  # noqa: E402
    Agent,
    AgentError,
    AgentExecutionConfig,
    JsonFileTraceSink,
)
from repo_intelligence.context_engine import (  # noqa: E402
    ContextBudget,
    ContextEngine,
)
from repo_intelligence.llm import (  # noqa: E402
    LLMError,
    LLMResponse,
    Message,
    StructuredOutputValidationError,
    ToolCall,
)
from repo_intelligence.llm.config import load_env  # noqa: E402
from repo_intelligence.llm.openai_client import (  # noqa: E402
    DEFAULT_MODEL,
    OpenAILLMClient,
)
from repo_intelligence.memory import (  # noqa: E402
    MemoryContext,
    MemoryManager,
    MemoryOwner,
    MemoryScope,
)
from repo_intelligence.multiagent import (  # noqa: E402
    ArchitectureFinding,
    ContractError,
    InvestigationReport,
    RULE_ORDER,
    TaskPayload,
)
from repo_intelligence.multiagent.permissions import (  # noqa: E402
    restricted_dispatcher,
    tool_definitions_for,
)
from repo_intelligence.orchestration import (  # noqa: E402
    BUDGET_EXCEEDED,
    COMPLETED,
    PLAN_FAILED,
    OrchestrationConfig,
    Orchestrator,
    assign_roles,
    is_terminal,
)
from repo_intelligence.planning.models import (  # noqa: E402
    InvestigationPlan,
    PlanStep,
)
from repo_intelligence.planning.planner import (  # noqa: E402
    build_planning_request,
    plan_from_response,
)
from repo_intelligence.planning.validation import (  # noqa: E402
    PlanValidationError,
    validate_plan,
)
from repo_intelligence.tools.read_file import read_file  # noqa: E402
from repo_intelligence.tools.search_repository import (  # noqa: E402
    search_repository,
)

from repo_intelligence.planning.planner import PLANNING_SYSTEM_INSTRUCTION  # noqa: E402
from repo_intelligence.repository_query import QueryRouter
import re
from repo_intelligence.agent import (
    Agent,
    AgentError,
    AgentExecutionConfig,
    JsonFileTraceSink,
)
from repo_intelligence.agents import (
    ARCHITECTURE_TOOLS,
    FLOW_DEPENDENCY_TOOLS,
    ArchitectureInvestigator,
    ArchitectureInvestigatorError,
    ArchitectureReport,
    ArchitectureVisualizationAgent,
    ArchitectureVisualizationAgentError,
    DiagramScope,
    DiagramSpec,
    FlowDependencyInvestigator,
    FlowDependencyInvestigatorError,
    FlowDependencyReport,
    RunSummaryContext,
    VisualizationArtifactError,
    validate_diagram_spec,
    verify_artifacts,
    write_visualization_artifacts,
)
from repo_intelligence.artifacts import (
    PRIMARY_SNAPSHOT_ID,
    PROJECT_ROOT,
    REPOSITORY_DIR_NAME,
    default_artifacts_root,
    repository_dir,
    snapshot_path,
)
from repo_intelligence.context.models import display_path as _project_display_path
from repo_intelligence.context_engine import ContextBudget, ContextEngine
from repo_intelligence.evidence.validator import gather_evidence
from repo_intelligence.llm import (
    LLMError,
    LLMResponse,
    StructuredOutputValidationError,
)
from repo_intelligence.llm.config import load_env
from repo_intelligence.llm.openai_client import DEFAULT_MODEL, OpenAILLMClient
from repo_intelligence.memory import MemoryContext, MemoryManager, MemoryOwner, MemoryScope
from repo_intelligence.multiagent import ArchitectureFinding, ContractError, TaskPayload
from repo_intelligence.multiagent.permissions import (
    restricted_dispatcher,
    tool_definitions_for,
)
from repo_intelligence.offline_investigation import (
    AnalystModel,
    InvestigatorModel,
    structured_response,
    synthesize_answer,
)
# -- exit codes (the CLI's contract) ---------------------------------------

EXIT_SUCCESS = 0
EXIT_INVESTIGATION_FAILED = 1
EXIT_USAGE = 2
EXIT_VALIDATION_FAILED = 3
EXIT_INTERRUPTED = 130

# -- bounds ----------------------------------------------------------------

MAX_CONCLUSIONS = 10
MAX_EVIDENCE = 12
MAX_UNCERTAINTIES = 8

SUBJECT = "repository-query"

DEFAULT_BUDGET = ContextBudget(max_items=64, max_characters=6000)

# M17 global execution budget — the mandated 64 tool calls, passed
# explicitly at THIS composition root (the Agent's own default of 8
# still governs every other construction in the project).
#
# The same configuration opts the workflow's agents into the boundary /
# finalization lifecycle, AND into the bounded investigation lifecycle:
# crossing the 8-call soft target opens the decision phase (reassess
# → finalize, or exactly ONE targeted call → reassess → finalize). This
# normally caps a healthy investigator at 9 calls, so up to five
# investigator steps plus one retry and deterministic validation fit
# inside the unchanged 64-call whole-run ceiling. An execution boundary
# is a LOCAL status (``incomplete``), never a workflow failure.
M17_MAX_TOOL_CALLS = 64
M17_SOFT_TOOL_CALL_TARGET = 8
M17_EXECUTION_CONFIG = AgentExecutionConfig(
    max_tool_calls=M17_MAX_TOOL_CALLS,
    soft_tool_call_target=M17_SOFT_TOOL_CALL_TARGET,
    finalize_on_boundary=True,
    enforce_soft_target=True,
)
# The matching whole-run hard bound remains 64 (rather than the
# orchestrator's smaller default of 32). The soft target above, not a
# larger hard budget, keeps multi-step investigations within this bound.
# Every other construction keeps the shipped defaults.
M17_ORCHESTRATION_CONFIG = OrchestrationConfig(
    max_total_tool_calls=M17_MAX_TOOL_CALLS,
)

# Offline scope label: a visibly synthetic commit (only reached for a
# repository that carries neither Git metadata nor an M1 snapshot
# record; the pinned subject always reports its real commit).
OFFLINE_COMMIT = "c" * 40

# -- the pinned subject repository this project analyzes -------------------
#
# repository-intelligence analyzes ITS SUBJECT, never itself: M1's own
# checkout of the ADOP repository, stored under ``artifacts/`` by the
# existing snapshot layout (``artifacts.py`` already owns the id).
# No clone happens here — the snapshot is the authority.
SUBJECT_SNAPSHOT_ID = PRIMARY_SNAPSHOT_ID
SUBJECT_COMMIT = "ef11dac8618b6344d6a5e4add6ef51cd2811d7e0"

# ``RepositoryQueryResult.termination_reason`` for a subject snapshot
# that could not be verified — the CLI prints the standing ERROR block
# for it and no investigation is executed.
SNAPSHOT_VERIFICATION_FAILED = "SNAPSHOT_VERIFICATION_FAILED"

# ``RepositoryQueryResult.termination_reason`` for ``--live`` when the
# configured provider cannot be constructed (no key, no client
# package): an explicit request for live mode NEVER falls back to the
# offline model — it fails, loudly, with these two lines first.
LIVE_CONFIGURATION_UNAVAILABLE = "LIVE_CONFIGURATION_UNAVAILABLE"

LIVE_FAILURE_MESSAGE = (
    "Live mode requested but the configured LLM provider is "
    "unavailable.\nSet the required configuration and retry."
)

# Files a repository root is searched for — the first readable one is
# the investigation target and the entry point every read starts from.
# The pinned ADOP snapshot reaches README.md (its root carries no
# ``agents/`` or ``src/`` tree); the MCP configuration, the registries
# and the agent definitions are found by the investigation itself.
READ_CANDIDATES: Tuple[str, ...] = (
    "agents/collector.py",
    "tools/fetch_record.py",
    "agents/researcher.py",
    "src/producer.py",
    "src/api.py",
    "src/service.py",
    "README.md",
    "main.py",
    "src/main.py",
)

# The runner labels an investigator step inside its question as
# "step: <id> — <goal>" (em dash U+2014) on its own line; the goal
# rides along so the router can fall back to it for a live planner's
# ids.  MULTILINE because the label is not the whole question.
_STEP_LABEL = re.compile("^step: (\\S+) \u2014 (.*)$", re.MULTILINE)

# The investigators' own statement form ends in its classification
# ("... [observed]"); parse it back so evidence can be annotated with
# the classification of the statement that cites it — never re-typed.
_STATEMENT_CLASS = re.compile("\\s*\\[([a-z_]+)\\]\\s*$")
RULE = "=" * 72
SECTION = "-" * 52

EXIT_SUCCESS = 0
EXIT_DEMO_FAILED = 1
EXIT_EXECUTION_FAILED = 6


In [ ]:
# -- printing helpers ------------------------------------------------------


def _clip(text: Optional[str], limit: int = 100) -> str:
    text = " ".join((text or "").split())
    if len(text) <= limit:
        return text
    return text[: limit - 1] + "\u2026"


def _header(title: str, stream) -> None:
    print(RULE, file=stream)
    print(title, file=stream)
    print(RULE, file=stream)

In [ ]:
from typing import Any

from pydantic import BaseModel, Field


class AgentAssignments(BaseModel):
    """Maps each plan-step ID to its assigned agent name."""

    agent_assignments: dict[str, str] = Field(default_factory=dict)

In [ ]:
class RepoIntelligenceState(TypedDict):
    objective:str
    plan: InvestigationPlan
    role_assignment:List[RoleAssignment]
    agent_assignments:AgentAssignments
    orchestration_run: Any
    execution_status: str
    execution_error: str

In [ ]:
def planner_node(state: RepoIntelligenceState) -> dict:
    plan = get_plan(state["objective"])
    if isinstance(plan, int):
        raise RuntimeError(f"Planner failed with code {plan}")
    return {"plan": plan}

In [ ]:
def role_assigner_node(state: RepoIntelligenceState):
    #stream = stream or sys.stdout
    assignments = assign_roles(state['plan'].steps)
    for item in assignments:
        print(
            "  {}  {:<12} {}".format(
                item.step_id, item.role, _clip(item.reason)
            ),
            #file=stream,
        )
    roles = [item.role for item in assignments]
    ok = (
        len(assignments) == len(state['plan'].steps)
        and roles[0] == "INVESTIGATOR"
        and roles[-1] == "ANALYST"
        and all(item.reason.strip() for item in assignments)
    )
    print(
        "roles: {} investigator / {} analyst".format(
            roles.count("INVESTIGATOR"), roles.count("ANALYST")
        ),
        #file=stream,
    )
    print("roles condition: {}".format("yes" if ok else "NO"))
    return {"role_assignment": assignments}


In [ ]:
from pathlib import Path

from repo_intelligence.agent import JsonFileTraceSink
from repo_intelligence.context_engine import ContextEngine
from repo_intelligence.llm.config import load_env
from repo_intelligence.llm.openai_client import OpenAILLMClient
from repo_intelligence.orchestration import Orchestrator
from repo_intelligence.repository_query import (
    QueryRouter,
    build_memory_manager,
    build_memory_context,
    build_engine,
    build_architect,
    build_flow_investigator,
    build_visualizer,
    default_subject_repository,
    resolve_subject_commit,
    live_bindings,
    default_artifacts_root,
    M17_ORCHESTRATION_CONFIG,
)

# ------------------------------------------------------------
# 1. Load configuration and locate the pinned subject repository
# ------------------------------------------------------------

load_env()

repository_root = default_subject_repository()

if not repository_root.is_dir():
    raise FileNotFoundError(
        f"Pinned subject repository not found: {repository_root}"
    )

current_commit, commit_source = resolve_subject_commit(repository_root)

if not current_commit:
    raise RuntimeError(
        f"Could not resolve the subject repository commit: {repository_root}"
    )

repository_name = str(repository_root)

print(f"Repository: {repository_root}")
print(f"Commit:     {current_commit}")
print(f"Commit source: {commit_source}")

# ------------------------------------------------------------
# 2. Initialize shared infrastructure
# ------------------------------------------------------------

memory_manager = build_memory_manager()

memory_context = build_memory_context(
    repository=repository_name,
    current_commit=current_commit,
)

context_engine = build_engine(repository_name)

trace_sink = JsonFileTraceSink()

tool_bindings = live_bindings(
    repository_root,
    output_root=default_artifacts_root(),
)

# ------------------------------------------------------------
# 3. Initialize the investigator model clients
# ------------------------------------------------------------

architect_llm = OpenAILLMClient()
flow_llm = OpenAILLMClient()
visualizer_llm = OpenAILLMClient()

# ------------------------------------------------------------
# 4. Build the actual investigator instances
# ------------------------------------------------------------

architect = build_architect(
    llm=architect_llm,
    trace_sink=trace_sink,
    memory_manager=memory_manager,
    memory_context=memory_context,
    context_engine=context_engine,
    bindings=tool_bindings,
)

flow_investigator = build_flow_investigator(
    llm=flow_llm,
    trace_sink=trace_sink,
    memory_manager=memory_manager,
    memory_context=memory_context,
    context_engine=context_engine,
    bindings=tool_bindings,
)

# ------------------------------------------------------------
# 5. Construct ONE reusable QueryRouter
# ------------------------------------------------------------

router = QueryRouter(
    architecture=architect,
    flow=flow_investigator,
)

print("QueryRouter initialized.")
print(f"Architecture specialist: {type(architect).__name__}")
print(f"Flow specialist:         {type(flow_investigator).__name__}")

# ------------------------------------------------------------
# 6. Build visualization analyst
# ------------------------------------------------------------

visualizer = build_visualizer(
    llm=visualizer_llm,
    trace_sink=trace_sink,
    memory_manager=memory_manager,
    memory_context=memory_context,
    context_engine=context_engine,
)

print(f"Visualization specialist: {type(visualizer).__name__}")

# ------------------------------------------------------------
# 7. Construct orchestrator using M17 configuration
# ------------------------------------------------------------

orchestrator = Orchestrator(
    router,
    visualizer,
    trace_sink=trace_sink,
    config=M17_ORCHESTRATION_CONFIG,
)

print("Orchestrator initialized.")

In [ ]:

def query_router_node(state: RepoIntelligenceState) -> dict:
    """Route investigator steps through QueryRouter and analysts to visualizer."""

    steps = state["plan"].steps
    role_assignments = state["role_assignment"]

    step_ids = [step.id for step in steps]

    if len(step_ids) != len(set(step_ids)):
        raise ValueError("The plan contains duplicate step IDs.")

    role_by_step = {}

    for item in role_assignments:
        if item.step_id in role_by_step:
            raise ValueError(
                f"Duplicate role assignment for step {item.step_id!r}"
            )
        role_by_step[item.step_id] = item.role

    if set(role_by_step) != set(step_ids):
        raise ValueError(
            "Role assignments must match the plan step IDs. "
            f"Plan IDs: {step_ids}; role IDs: {list(role_by_step)}"
        )

    assignments: dict[str, str] = {}

    for step in steps:
        step_id = step.id
        role = role_by_step[step_id]

        if role == "ANALYST":
            agent_name = "visualizer"

        elif role == "INVESTIGATOR":
            # Reuse QueryRouter's existing routing logic.
            specialist = router._resolve(step_id, step.goal)

            if specialist is architect:
                agent_name = "architect"

            elif specialist is flow_investigator:
                agent_name = "flow_investigator"

            else:
                raise ValueError(
                    f"QueryRouter returned an unexpected specialist "
                    f"for step {step_id!r}: {specialist!r}"
                )

        else:
            raise ValueError(
                f"Unsupported role {role!r} for step {step_id!r}"
            )

        assignments[step_id] = agent_name
        print(f"  {step_id}: {role} -> {agent_name}")

    return {
        "agent_assignments": AgentAssignments(
            agent_assignments=assignments
        )
    }


In [ ]:

test_state = {
    "plan": InvestigationPlan(
        objective="How do agents and MCP servers interact?",
        steps=[
            PlanStep(
                id="S1",
                goal="Identify the main architecture components",
            ),
            PlanStep(
                id="S2",
                goal="Trace MCP server interactions and dependencies",
            ),
            PlanStep(
                id="S3",
                goal="Synthesize the findings into a diagram",
            ),
        ],
    ),
    "role_assignment": [
        RoleAssignment(
            step_id="S1",
            role="INVESTIGATOR",
            reason="Collect architecture evidence",
        ),
        RoleAssignment(
            step_id="S2",
            role="INVESTIGATOR",
            reason="Trace interaction paths",
        ),
        RoleAssignment(
            step_id="S3",
            role="ANALYST",
            reason="Synthesize the findings",
        ),
    ],
}

routing_result = query_router_node(test_state)
print(routing_result["agent_assignments"])


In [ ]:
from repo_intelligence.repository_query import (
    default_subject_repository,
    resolve_read_target,
    verify_subject_snapshot,
    read_snapshot_metadata,
    OFFLINE_COMMIT,
)

repository_root, target, resolve_error = resolve_read_target()

if resolve_error:
    raise RuntimeError(resolve_error)

verified_commit, commit_source, verification_error = (
    verify_subject_snapshot(repository_root)
)

if verification_error:
    raise RuntimeError(verification_error)

metadata = read_snapshot_metadata(repository_root)

repository_label = (
    metadata.get("name")
    or repository_root.name
    or str(repository_root)
)

current_commit = verified_commit or OFFLINE_COMMIT

print("Repository root:", repository_root)
print("Repository scope:", repository_label)
print("Commit scope:", current_commit)
print("Commit source:", commit_source)
print("Readable target:", target)

In [ ]:
from repo_intelligence.multiagent import TaskPayload
from repo_intelligence.orchestration import Orchestrator
from repo_intelligence.repository_query import QueryRouter

def orchestrator_node(state: RepoIntelligenceState) -> dict:
    """Execute the validated plan using the existing orchestration runtime."""

    plan = state["plan"]

    task = TaskPayload(
        question=state["objective"],
        repository_scope=str(repository_root),
        commit_scope=current_commit,
    )

    orchestrator = Orchestrator(
        investigator=router,
        analyst=visualizer,
        trace_sink=trace_sink,
    )

    run = orchestrator.execute(task, plan)

    return {
        "orchestration_run": run,
        "execution_status": run.status,
        "execution_error": run.termination_reason or "",
    }

In [ ]:

from IPython.display import Image, display
from langchain_core.runnables.graph import MermaidDrawMethod
# Build graph
builder = StateGraph(RepoIntelligenceState)
builder.add_node("planner", planner_node)
builder.add_node("assign_role", role_assigner_node)
builder.add_node("query_router", query_router_node)
builder.add_node("orchestrator", orchestrator_node)
builder.add_edge(START, "planner")
builder.add_edge("planner", "assign_role")
builder.add_edge("assign_role", "query_router")
builder.add_edge("query_router", "orchestrator")
builder.add_edge("orchestrator", END)

# Compile graph
graph = builder.compile()

#display(Image(graph.get_graph().draw_mermaid_png()))
print(graph.get_graph().draw_mermaid())


In [ ]:
result = graph.invoke({
    "objective": "How do agents and MCP servers interact?"
})
print(f"========================FINAL RESULT=================================")
print(result['agent_assignments'])